# **MODEL 1: Multimodal Integration - Late Fusion (EfficientNetB1 + Metadata MLP)**

**Objective:**
Inclusion of **clinical metadata**
- **Age**
- **Sex**
- **Atomic location** of the lesion
<br>

**Methodology:**
| Variable | Strategy | NA Treatment |
|----------|-----------|--------------------|
| Age | Normalization (min-max) | Imputation by median |
| Sex | One-hot encoding | Category "unknown" |
| Location | One-hot encoding (15 sítios) | Category "unknown" |

### Imports

In [1]:
import os, sys, json
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix

### **Data** Configuration and loading

In [2]:
if os.getcwd().endswith('models'):
    os.chdir('..')

In [3]:
BASE_PATH = "./data"
TRAIN_CSV = os.path.join(BASE_PATH, "augmented_metadata.csv")
VAL_CSV = os.path.join(BASE_PATH, "val_split.csv")
TEST_CSV = os.path.join(BASE_PATH, "test_split.csv")

MODEL_SAVE_PATH = "./models/best_model_m1.keras"


In [4]:
train_df = pd.read_csv(TRAIN_CSV)
val_df = pd.read_csv(VAL_CSV)
test_df = pd.read_csv(TEST_CSV)

## Metadata preprocessing

Clean and encode the clinical features (Age, Sex, Localization) into a fixed-length vector before the split.

In [5]:
def preprocess_metadata(df, scaler=None, encoder=None, is_training=True):
    """Clean and encode clinical features into a fixed-length vector."""
    df = df.copy()

    # Impute missing values
    df['age'] = df['age'].fillna(df['age'].median())
    df['sex'] = df['sex'].fillna('unknown')
    df['localization'] = df['localization'].fillna('unknown')

    # Scale age
    if is_training:
        scaler = MinMaxScaler()
        age_scaled = scaler.fit_transform(df[['age']])
    else:
        age_scaled = scaler.transform(df[['age']])

    # One-hot encode categorical columns
    cat_cols = ['sex', 'localization']
    if is_training:
        encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
        encoded_cats = encoder.fit_transform(df[cat_cols])
    else:
        encoded_cats = encoder.transform(df[cat_cols])

    meta_vectors = np.hstack([age_scaled, encoded_cats])
    return meta_vectors, scaler, encoder

In [6]:
train_meta, scaler, encoder = preprocess_metadata(train_df, is_training=True)
val_meta, _, _ = preprocess_metadata(val_df, scaler, encoder, is_training=False)
test_meta, _, _ = preprocess_metadata(test_df, scaler, encoder, is_training=False)

In [7]:
# Verification
META_DIM = train_meta.shape[1]
print(f"Metadata vector dimension: {META_DIM}")

Metadata vector dimension: 19


## 4. Multimodal Data Pipeline

In [8]:
def load_multimodal_item(path, meta, label):
    img = tf.io.read_file(path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, [240, 240])                              # EfficientNetB1 native size
    img = tf.cast(img, tf.float32)
    img = tf.keras.applications.efficientnet.preprocess_input(img)
    return {"image_input": img, "meta_input": meta}, label


In [9]:
def create_ds(df, meta, shuffle=False):
    ds = tf.data.Dataset.from_tensor_slices((
        df['image_path'].values,
        meta,
        df['dx_encoded'].values.astype(np.int32)
    )).map(load_multimodal_item, num_parallel_calls=tf.data.AUTOTUNE)

    if shuffle:
        ds = ds.shuffle(buffer_size=len(df), reshuffle_each_iteration=True)

    return ds.batch(32).prefetch(tf.data.AUTOTUNE)

In [10]:
train_ds = create_ds(train_df, train_meta)
val_ds = create_ds(val_df, val_meta)
test_ds = create_ds(test_df, test_meta)

## 5. Architecture Late Fusion: EfficientNetB1 + Metadata MLP 

This builds the two branches and concatenates them into the final classification head. Establishing correct connections between EfficientNet and the MLP

In [11]:
# Image stream (EfficientNetB1)
base_model = tf.keras.applications.EfficientNetB1(
    include_top=False,
    weights='imagenet',
    input_shape=(240, 240, 3)
)

In [12]:
# Phase 1: freeze the pretrained base entirely
base_model.trainable = False

image_input = layers.Input(shape=(240, 240, 3), name="image_input")
x = base_model(image_input, training=False)   # training=False keeps BN in inference mode while frozen
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(128, activation='relu')(x)   # reduced to 128 to match meta branch
image_features = layers.Dropout(0.4)(x)       # dropout after dense

In [13]:
# Metadata Stream (MLP)
meta_input = layers.Input(shape=(META_DIM,), name="meta_input")
y = layers.Dense(64, activation='relu')(meta_input)
y = layers.Dropout(0.3)(y)                    # dropout in MLP
y = layers.Dense(128, activation='relu')(y)   # raised to 128 to match image branch
meta_features = layers.Dropout(0.3)(y)        # dropout after final MLP dense

In [14]:
# Concatenation (The Fusion Point)
combined = layers.Concatenate()([image_features, meta_features])  # 256-d (128 + 128)
combined = layers.Dense(128, activation='relu')(combined)          # optional fusion FC
combined = layers.Dropout(0.3)(combined)
final_output = layers.Dense(7, activation='softmax')(combined)

model = models.Model(inputs=[image_input, meta_input], outputs=final_output)

Training

In [15]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=[
        'accuracy',
        tf.keras.metrics.AUC(name='auc', multi_label=False),
    ]
)

In [16]:
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ image_input         │ (None, 240, 240,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ meta_input          │ (None, 19)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ efficientnetb1      │ (None, 8, 8,      │  6,575,239 │ image_input[0][0] │
│ (Functional)        │ 1280)             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 64)        │      1,280 │ meta_input[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 1280)      │          0 │ efficientnetb1[0… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 64)        │          0 │ dense_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 128)       │    163,968 │ global_average_p… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 128)       │      8,320 │ dropout_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 128)       │          0 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_2 (Dropout) │ (None, 128)       │          0 │ dense_2[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 256)       │          0 │ dropout[0][0],    │
│ (Concatenate)       │                   │            │ dropout_2[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_3 (Dense)     │ (None, 128)       │     32,896 │ concatenate[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_3 (Dropout) │ (None, 128)       │          0 │ dense_3[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_4 (Dense)     │ (None, 7)         │        903 │ dropout_3[0][0]   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 6,782,606 (25.87 MB)

 Trainable params: 207,367 (810.03 KB)

 Non-trainable params: 6,575,239 (25.08 MB)

## Phase 1 — Train classification head only (base frozen)

EarlyStopping and ModelCheckpoint ensure we save the best epoch and stop before overfitting. ReduceLROnPlateau decays LR when val_loss plateaus.

In [ ]:
callbacks_phase1 = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=5,
        restore_best_weights=True,
        verbose=1
    ),
    tf.keras.callbacks.ModelCheckpoint(
        filepath=MODEL_SAVE_PATH,
        monitor='val_loss',
        save_best_only=True,
        verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=3,
        min_lr=1e-6,
        verbose=1
    ),
]


In [ ]:
print("Phase 1: training head (base frozen)")
history_phase1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20,
    callbacks=callbacks_phase1,
    class_weight=class_weight_dict,   # FIX 5
    verbose=1
)

## Phase 2 — Fine-tune: unfreeze base and train end-to-end

Unfreeze the entire EfficientNetB1 and re-compile with amuch smaller learning rate to avoid destroying the pretrained weights.

In [ ]:
base_model.trainable = True

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),   # 100x smaller LR
    loss='sparse_categorical_crossentropy',
    metrics=[
        'accuracy',
        tf.keras.metrics.AUC(name='auc', multi_label=False),
    ]
)

In [ ]:
callbacks_phase2 = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=7,
        restore_best_weights=True,
        verbose=1
    ),
    tf.keras.callbacks.ModelCheckpoint(
        filepath=MODEL_SAVE_PATH,
        monitor='val_loss',
        save_best_only=True,
        verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=3,
        min_lr=1e-7,
        verbose=1
    ),
]

In [ ]:
print("Phase 2: fine-tuning (base unfrozen)")
history_phase2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=30,
    callbacks=callbacks_phase2,
    class_weight=class_weight_dict,   # FIX 5
    verbose=1
)

## Evaluation on Test Set

classification_report gives per-class precision/recall/F1,which is the correct metric for imbalanced multiclass classification.

In [ ]:
# Load best checkpoint before evaluating
model.load_weights(MODEL_SAVE_PATH)

print("Test set evaluation")
test_loss, test_acc, test_auc = model.evaluate(test_ds, verbose=1)
print(f"\nTest Loss : {test_loss:.4f}")
print(f"Test Acc  : {test_acc:.4f}")
print(f"Test AUC  : {test_auc:.4f}")

In [ ]:
# Per-class metrics
y_true = test_df['dx_encoded'].values.astype(np.int32)

In [ ]:
# Collect predictions across all batches
y_pred_probs = model.predict(test_ds, verbose=1)
y_pred = np.argmax(y_pred_probs, axis=1)

In [ ]:
# Decode label names
label_map = {v: k for k, v in
             dict(enumerate(sorted(train_df['dx'].unique()))).items()} \
            if 'dx' in train_df.columns else None

target_names = [label_map[i] for i in sorted(label_map.keys())] \
               if label_map else [str(i) for i in range(7)]

print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=target_names))

print("\nConfusion Matrix:")
print(confusion_matrix(y_true, y_pred))